In [ ]:
# 필요한 라이브러리 임포트
import os
import glob
import json
import yaml
import copy
import random
import warnings
import numpy as np
import cv2
import torch
import tqdm
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# MMDetection 관련 임포트
from mmengine.config import Config
from mmengine.runner import Runner
from mmdet.apis import init_detector, inference_detector
from mmdet.registry import DATASETS, MODELS
from mmdet.structures import DetDataSample
from mmengine.structures import InstanceData

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device:", device)
warnings.filterwarnings('ignore')

In [ ]:
# 파라미터 및 데이터 경로 설정
with open('utils/args.yaml', errors='ignore') as f:
    params = yaml.safe_load(f)

label_dir = '../../data/HnE_cell_detect/total_data/labels/'
image_dir = '../../data/HnE_cell_detect/total_data/images/'

# 데이터 로드
label_files = sorted(glob.glob(os.path.join(label_dir, '*.json')))
filenames = []
labels = []

for label_file in label_files:
    with open(label_file) as f:
        data1 = json.load(f)
    img_path = os.path.join(image_dir, data1['file_name'])
    if os.path.exists(img_path):
        filenames.append(img_path)
        temp_labels = []
        for i in range(len(data1["cordinates"])):
            # 너무 큰 박스는 제외
            if data1["cordinates"][i][3] > 50 or data1["cordinates"][i][4] > 50:
                continue
            temp_labels.append(data1["cordinates"][i])
        labels.append(temp_labels)

print(f"총 이미지 수: {len(filenames)}")
print(f"첫 번째 이미지의 라벨 수: {len(labels[0]) if labels else 0}")

In [ ]:
# Point Detection용 커스텀 데이터셋 클래스
class PointDetectionDataset(torch.utils.data.Dataset):
    """Point Detection을 위한 커스텀 데이터셋"""
    def __init__(self, filenames, labels, input_size=512, augment=False, params=None):
        self.filenames = filenames
        self.labels = labels
        self.input_size = input_size
        self.augment = augment
        self.params = params
        self.n = len(self.filenames)
        self.indices = range(self.n)
        
    def __len__(self):
        return self.n * 5 if self.augment else self.n
    
    def __getitem__(self, index):
        index = index % self.n
        index = self.indices[index]
        temp_label = copy.deepcopy(self.labels[index])
        
        # 이미지 로드 및 크롭
        image, crop_index = self.load_image(index)
        crop_y, crop_x = crop_index
        
        # Point annotation 생성 (중심점만 사용)
        points = []
        classes = []
        
        for i in range(len(temp_label)):
            x = temp_label[i][2]
            y = temp_label[i][1]
            w = temp_label[i][4]
            h = temp_label[i][3]
            
            # 중심점 계산
            center_x = x + w / 2
            center_y = y + h / 2
            
            # 크롭된 영역 내에 중심점이 있는지 확인
            if (center_x >= crop_x and center_y >= crop_y and 
                center_x <= crop_x + self.input_size and 
                center_y <= crop_y + self.input_size):
                # 절대 픽셀 좌표로 변환 (0~512 범위)
                abs_x = center_x - crop_x
                abs_y = center_y - crop_y
                
                points.append([abs_x, abs_y])
                classes.append(temp_label[i][0] - 1)  # 클래스 인덱스 (0-based)
        
        points = np.array(points) if len(points) > 0 else np.zeros((0, 2))
        classes = np.array(classes) if len(classes) > 0 else np.zeros((0,))
        
        # Augmentation (절대 좌표 기준)
        if self.augment and self.params:
            # Flip up-down
            if random.random() < self.params.get('flip_ud', 0.5):
                image = np.flipud(image).copy()
                if len(points) > 0:
                    points[:, 1] = self.input_size - points[:, 1]
            
            # Flip left-right
            if random.random() < self.params.get('flip_lr', 0.5):
                image = np.fliplr(image).copy()
                if len(points) > 0:
                    points[:, 0] = self.input_size - points[:, 0]
        
        # 이미지를 (C, H, W) 형식으로 변환
        image = image.transpose((2, 0, 1))
        
        return {
            'image': torch.from_numpy(image).float(),
            'points': torch.from_numpy(points).float(),
            'classes': torch.from_numpy(classes).long(),
        }
    
    def load_image(self, i):
        """이미지를 로드하고 크롭/패딩"""
        image = cv2.imread(self.filenames[i])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        h, w = image.shape[:2]
        r = self.input_size / min(h, w)
        
        # 이미지가 input_size보다 큰 경우 랜덤 크롭
        if r < 1:
            max_h = max(0, h - self.input_size)
            max_w = max(0, w - self.input_size)
            h1 = random.randint(0, max_h) if max_h > 0 else 0
            w1 = random.randint(0, max_w) if max_w > 0 else 0
            image = image[h1:h1 + self.input_size, w1:w1 + self.input_size]
        else:
            # 이미지가 input_size보다 작은 경우 패딩
            h1 = 0
            w1 = 0
            pad_image = np.ones((self.input_size, self.input_size, 3), dtype=np.uint8) * 255
            pad_image[:min(h, self.input_size), :min(w, self.input_size), :] = image[:min(h, self.input_size), :min(w, self.input_size), :]
            image = pad_image
        
        return image, (h1, w1)


# 데이터셋 분할
split = [0.9, 0.1]
x_train, x_val, y_train, y_val = train_test_split(
    filenames, labels, test_size=0.1, random_state=42, shuffle=True
)

# 데이터셋 생성
train_dataset = PointDetectionDataset(x_train, y_train, input_size=512, augment=True, params=params)
val_dataset = PointDetectionDataset(x_val, y_val, input_size=512, augment=False, params=params)

print(f"훈련 데이터셋 크기: {len(train_dataset)}")
print(f"검증 데이터셋 크기: {len(val_dataset)}")

In [ ]:
# RTMDet 설정 파일 로드 및 수정
config_path = 'configs/rtmdet/rtmdet_s_cell_detection.py'
cfg = Config.fromfile(config_path)

# Point detection을 위한 모델 수정
# RTMDet을 Point detection으로 수정하기 위해 bbox_head를 조정
cfg.model.bbox_head.update(dict(
    type='RTMDetSepBNHead',
    num_classes=6,
    in_channels=128,
    stacked_convs=2,
    feat_channels=128,
    anchor_generator=dict(
        type='MlvlPointGenerator', offset=0, strides=[8, 16, 32]),
    bbox_coder=dict(type='DistancePointBBoxCoder'),
    loss_cls=dict(
        type='QualityFocalLoss',
        use_sigmoid=True,
        beta=2.0,
        loss_weight=1.0),
    loss_bbox=dict(type='GIoULoss', loss_weight=2.0),
))

# 학습 설정
cfg.work_dir = '../../model/HnE_cell_detection/rtmdet/'
cfg.train_dataloader = dict(
    batch_size=16,
    num_workers=4,
    persistent_workers=True,
    sampler=dict(type='DefaultSampler', shuffle=True),
    dataset=dict(
        type='RepeatDataset',
        times=1,
        dataset=None  # 나중에 커스텀 데이터셋으로 대체
    )
)

cfg.val_dataloader = dict(
    batch_size=1,
    num_workers=2,
    persistent_workers=True,
    drop_last=False,
    sampler=dict(type='DefaultSampler', shuffle=False),
    dataset=None  # 나중에 커스텀 데이터셋으로 대체
)

cfg.optim_wrapper = dict(
    type='OptimWrapper',
    optimizer=dict(type='AdamW', lr=0.001, weight_decay=0.05),
    paramwise_cfg=dict(
        norm_decay_mult=0, bias_decay_mult=0, bypass_duplicate=True)
)

cfg.train_cfg = dict(
    type='EpochBasedTrainLoop',
    max_epochs=100,
    val_interval=10
)

cfg.val_cfg = dict(type='ValLoop')
cfg.test_cfg = dict(type='TestLoop')

# 학습률 스케줄러
cfg.param_scheduler = [
    dict(
        type='LinearLR', start_factor=0.001, by_epoch=False, begin=0, end=500),
    dict(
        type='MultiStepLR',
        begin=0,
        end=100,
        by_epoch=True,
        milestones=[70, 90],
        gamma=0.1)
]

# 체크포인트 저장 설정
cfg.default_hooks = dict(
    timer=dict(type='IterTimerHook'),
    logger=dict(type='LoggerHook', interval=50),
    param_scheduler=dict(type='ParamSchedulerHook'),
    checkpoint=dict(type='CheckpointHook', interval=10, max_keep_ckpts=3),
    sampler_seed=dict(type='DistSamplerSeedHook'),
    visualization=dict(type='DetVisualizationHook')
)

os.makedirs(cfg.work_dir, exist_ok=True)
print(f"작업 디렉토리: {cfg.work_dir}")
print(f"배치 크기: {cfg.train_dataloader['batch_size']}")
print(f"최대 에폭: {cfg.train_cfg['max_epochs']}")

In [ ]:
# Point Detection을 위한 커스텀 Loss 함수 (Hungarian Matching 방식)
from scipy.optimize import linear_sum_assignment

class PointDetectionLoss(torch.nn.Module):
    """Point Detection을 위한 Loss 함수 - Hungarian Matching 사용"""
    def __init__(self, cost_point=1.0, cost_class=1.0):
        super().__init__()
        self.cost_point = cost_point
        self.cost_class = cost_class
        
    def forward(self, pred_points, pred_classes, pred_conf, gt_points, gt_classes):
        """
        Args:
            pred_points: (B, N, 2) - 예측된 포인트 (N=1000)
            pred_classes: (B, N, num_classes) - 예측된 클래스 로짓
            pred_conf: (B, N) - 예측된 confidence
            gt_points: (B, M, 2) - Ground truth 포인트 (M은 실제 GT 수, 패딩 포함)
            gt_classes: (B, M) - Ground truth 클래스
        """
        batch_size = pred_points.shape[0]
        num_queries = pred_points.shape[1]  # 1000
        
        total_point_loss = 0
        total_class_loss = 0
        total_conf_loss = 0
        
        for b in range(batch_size):
            # GT에서 유효한 포인트만 선택 (패딩 제외)
            valid_mask = gt_classes[b] >= 0
            num_gt = valid_mask.sum().item()
            
            if num_gt == 0:
                # GT가 없는 경우: 모든 예측이 background (conf=0)
                target_conf = torch.zeros(num_queries, device=pred_conf.device)
                conf_loss = torch.nn.functional.binary_cross_entropy_with_logits(
                    pred_conf[b], target_conf, reduction='sum'
                )
                total_conf_loss += conf_loss / num_queries
                continue
            
            valid_gt_points = gt_points[b][valid_mask]  # (M', 2)
            valid_gt_classes = gt_classes[b][valid_mask]  # (M',)
            
            # Cost matrix 계산
            # Point distance cost: (N, M')
            point_cost = torch.cdist(pred_points[b], valid_gt_points, p=2)
            
            # Classification cost: (N, M')
            pred_probs = torch.softmax(pred_classes[b], dim=-1)  # (N, num_classes)
            class_cost = -pred_probs[:, valid_gt_classes].T  # (M', N) -> transpose -> (N, M')
            class_cost = class_cost.T
            
            # Total cost
            cost_matrix = self.cost_point * point_cost + self.cost_class * class_cost
            
            # Hungarian matching
            cost_matrix_np = cost_matrix.detach().cpu().numpy()
            pred_indices, gt_indices = linear_sum_assignment(cost_matrix_np)
            
            # Matched predictions
            matched_pred_points = pred_points[b][pred_indices]  # (M', 2)
            matched_pred_classes = pred_classes[b][pred_indices]  # (M', num_classes)
            matched_gt_points = valid_gt_points[gt_indices]  # (M', 2)
            matched_gt_classes = valid_gt_classes[gt_indices]  # (M',)
            
            # Point regression loss (L1)
            point_loss = torch.nn.functional.l1_loss(matched_pred_points, matched_gt_points, reduction='sum')
            total_point_loss += point_loss / num_gt
            
            # Classification loss
            class_loss = torch.nn.functional.cross_entropy(
                matched_pred_classes, matched_gt_classes, reduction='sum'
            )
            total_class_loss += class_loss / num_gt
            
            # Confidence loss
            # Matched predictions should have high confidence
            target_conf = torch.zeros(num_queries, device=pred_conf.device)
            target_conf[pred_indices] = 1.0
            conf_loss = torch.nn.functional.binary_cross_entropy_with_logits(
                pred_conf[b], target_conf, reduction='sum'
            )
            total_conf_loss += conf_loss / num_queries
        
        # 배치 평균
        avg_point_loss = total_point_loss / batch_size
        avg_class_loss = total_class_loss / batch_size
        avg_conf_loss = total_conf_loss / batch_size
        
        total_loss = avg_point_loss + avg_class_loss + avg_conf_loss
        
        return total_loss, avg_point_loss, avg_class_loss, avg_conf_loss


# 간단한 Point Detection 모델 (RTMDet 대신 간단한 버전)
class SimplePointDetector(torch.nn.Module):
    """간단한 Point Detection 모델"""
    def __init__(self, num_classes=6, max_detections=1000):
        super().__init__()
        self.num_classes = num_classes
        self.max_detections = max_detections
        
        # Backbone (간단한 CNN)
        self.backbone = torch.nn.Sequential(
            torch.nn.Conv2d(3, 64, 3, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2),
            torch.nn.Conv2d(64, 128, 3, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2),
            torch.nn.Conv2d(128, 256, 3, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2),
            torch.nn.Conv2d(256, 512, 3, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2),
        )
        
        # Global Average Pooling
        self.gap = torch.nn.AdaptiveAvgPool2d(1)
        
        # Point regression head
        self.point_head = torch.nn.Sequential(
            torch.nn.Linear(512, 256),
            torch.nn.ReLU(),
            torch.nn.Linear(256, max_detections * 2)
        )
        
        # Classification head
        self.class_head = torch.nn.Sequential(
            torch.nn.Linear(512, 256),
            torch.nn.ReLU(),
            torch.nn.Linear(256, max_detections * num_classes)
        )
        
        # Confidence head (Sigmoid 제거 - loss에서 with_logits 사용)
        self.conf_head = torch.nn.Sequential(
            torch.nn.Linear(512, 256),
            torch.nn.ReLU(),
            torch.nn.Linear(256, max_detections)
        )
        
    def forward(self, x):
        # Backbone
        features = self.backbone(x)
        features = self.gap(features)
        features = features.view(features.size(0), -1)
        
        # Predict points (절대 픽셀 좌표 0~512)
        points = self.point_head(features)
        points = points.view(-1, self.max_detections, 2)
        points = torch.sigmoid(points) * 512  # 0~512 범위로 스케일링
        
        # Predict classes
        classes = self.class_head(features)
        classes = classes.view(-1, self.max_detections, self.num_classes)
        
        # Predict confidence
        confidence = self.conf_head(features)
        
        return points, classes, confidence


# 모델 초기화
model = SimplePointDetector(num_classes=6, max_detections=1000).to(device)
criterion = PointDetectionLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.05)

print("모델 초기화 완료")
print(f"모델 파라미터 수: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"Max detections: {model.max_detections}")

In [ ]:
# DataLoader 생성
def collate_fn_point(batch):
    """Point detection을 위한 커스텀 collate 함수"""
    images = []
    points = []
    classes = []
    
    max_points = max([item['points'].shape[0] for item in batch]) if batch else 0
    
    for item in batch:
        images.append(item['image'])
        
        # 패딩 추가 (중요: classes는 -1로 초기화하여 valid class와 구분!)
        num_points = item['points'].shape[0]
        if num_points < max_points:
            padded_points = torch.zeros((max_points, 2))
            padded_classes = torch.full((max_points,), -1, dtype=torch.long)  # -1로 패딩!
            padded_points[:num_points] = item['points']
            padded_classes[:num_points] = item['classes']
            points.append(padded_points)
            classes.append(padded_classes)
        else:
            points.append(item['points'])
            classes.append(item['classes'])
    
    return {
        'images': torch.stack(images),
        'points': torch.stack(points),
        'classes': torch.stack(classes),
    }

# DataLoader 생성
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=4,
    collate_fn=collate_fn_point,
    drop_last=True
)

val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=2,
    collate_fn=collate_fn_point,
    drop_last=False
)

print(f"훈련 배치 수: {len(train_loader)}")
print(f"검증 배치 수: {len(val_loader)}")

In [ ]:
# 검증 함수
def compute_point_detection_metrics(model, val_loader, device, distance_threshold=25.0):
    """Point detection 메트릭 계산 (절대 픽셀 좌표 기준)"""
    model.eval()
    
    total_tp = 0
    total_fp = 0
    total_fn = 0
    total_gt = 0
    total_pred = 0
    
    with torch.no_grad():
        for batch in tqdm.tqdm(val_loader, desc='Validation'):
            images = batch['images'].to(device).float() / 255.0
            gt_points = batch['points'].to(device)
            gt_classes = batch['classes'].to(device)
            
            # 예측
            pred_points, pred_classes, pred_conf = model(images)
            
            # 배치 단위로 처리
            for i in range(images.size(0)):
                # Confidence threshold 적용 (logits를 sigmoid로 변환)
                pred_conf_sigmoid = torch.sigmoid(pred_conf[i])
                conf_mask = pred_conf_sigmoid > 0.5
                valid_pred_points = pred_points[i][conf_mask]
                valid_pred_classes = torch.argmax(pred_classes[i][conf_mask], dim=-1)
                
                # GT에서 유효한 포인트만 선택 (패딩 제외)
                valid_gt_mask = gt_classes[i] >= 0
                valid_gt_points = gt_points[i][valid_gt_mask]
                valid_gt_classes = gt_classes[i][valid_gt_mask]
                
                total_gt += len(valid_gt_points)
                total_pred += len(valid_pred_points)
                
                # 매칭
                if len(valid_pred_points) > 0 and len(valid_gt_points) > 0:
                    # 거리 계산
                    distances = torch.cdist(valid_pred_points, valid_gt_points)
                    
                    # 각 GT에 대해 가장 가까운 예측 찾기
                    matched_pred = torch.zeros(len(valid_gt_points), dtype=torch.bool)
                    matched_gt = torch.zeros(len(valid_gt_points), dtype=torch.bool)
                    
                    for j in range(len(valid_pred_points)):
                        if len(valid_gt_points) == 0:
                            break
                        
                        min_dist, min_idx = distances[j].min(dim=0)
                        
                        if min_dist < distance_threshold and not matched_gt[min_idx]:
                            # 클래스도 일치하는지 확인
                            if valid_pred_classes[j] == valid_gt_classes[min_idx]:
                                total_tp += 1
                                matched_gt[min_idx] = True
                            else:
                                total_fp += 1
                        else:
                            total_fp += 1
                    
                    total_fn += (~matched_gt).sum().item()
                else:
                    total_fn += len(valid_gt_points)
                    total_fp += len(valid_pred_points)
    
    # 메트릭 계산
    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
    recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    return precision, recall, f1_score


# AverageMeter 클래스 (손실 추적용)
class AverageMeter:
    """평균 계산을 위한 유틸리티 클래스"""
    def __init__(self):
        self.reset()
    
    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0
    
    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

print("검증 함수 준비 완료")

In [ ]:
# 시각화 함수
def visualize_point_detection(model, dataset, idx=0, conf_threshold=0.5, epoch=None, save_dir=None):
    """Point detection 결과 시각화"""
    model.eval()
    
    sample = dataset[idx]
    img = sample['image']
    gt_points = sample['points']
    gt_classes = sample['classes']
    
    # 2개의 subplot 생성
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
    
    # 이미지 표시 (0-255 범위로 변환)
    img_display = img.permute(1, 2, 0).cpu().numpy() / 255.0
    
    class_names = {
        0: "Neutrophil",
        1: "Epithelial",
        2: "Lymphocyte",
        3: "Plasma",
        4: "Eosinophil",
        5: "Connective tissue"
    }
    
    colors = ['orange', 'green', 'red', 'skyblue', 'blue', 'yellow']
    
    # Ground Truth 시각화
    ax1.imshow(img_display)
    ax1.set_title(f'Ground Truth - {len(gt_points)} points', fontsize=16, fontweight='bold')
    
    for i in range(len(gt_points)):
        if gt_classes[i] >= 0:  # 유효한 포인트만
            x_pixel, y_pixel = gt_points[i].tolist()  # 이미 절대 좌표
            class_id = gt_classes[i].item()
            ax1.scatter(x_pixel, y_pixel, s=100, marker='o', 
                       facecolors='none', edgecolors=colors[class_id], linewidths=2)
    
    ax1.axis('off')
    
    # Prediction 시각화
    ax2.imshow(img_display)
    
    with torch.no_grad():
        img_input = img.unsqueeze(0).to(device).float() / 255.0
        pred_points, pred_classes, pred_conf = model(img_input)
        
        # Confidence threshold 적용 (logits를 sigmoid로 변환)
        pred_conf_sigmoid = torch.sigmoid(pred_conf[0])
        conf_mask = pred_conf_sigmoid > conf_threshold
        valid_pred_points = pred_points[0][conf_mask]
        valid_pred_classes = torch.argmax(pred_classes[0][conf_mask], dim=-1)
        
        prediction_count = len(valid_pred_points)
        ax2.set_title(f'Prediction - {prediction_count} points', fontsize=16, fontweight='bold')
        
        for i in range(len(valid_pred_points)):
            x_pixel, y_pixel = valid_pred_points[i].tolist()  # 이미 절대 좌표
            class_id = valid_pred_classes[i].item()
            ax2.scatter(x_pixel, y_pixel, s=100, marker='o',
                       facecolors='none', edgecolors=colors[class_id], linewidths=2)
    
    ax2.axis('off')
    
    # 전체 제목
    if epoch is not None:
        fig.suptitle(f'Point Detection - Epoch {epoch}, Sample {idx+1}',
                    fontsize=18, fontweight='bold', y=0.95)
    
    # 범례 추가
    legend_elements = [
        patches.Patch(color='orange', label='Neutrophil'),
        patches.Patch(color='green', label='Epithelial'),
        patches.Patch(color='red', label='Lymphocyte'),
        patches.Patch(color='skyblue', label='Plasma'),
        patches.Patch(color='blue', label='Eosinophil'),
        patches.Patch(color='yellow', label='Connective tissue'),
    ]
    fig.legend(handles=legend_elements, loc='lower center', ncol=3,
              bbox_to_anchor=(0.5, 0.02), fontsize=12)
    
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.15, top=0.85)
    
    # 저장
    if save_dir and epoch:
        save_path = os.path.join(save_dir, f'point_detection_epoch_{epoch}.png')
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"✅ 이미지 저장: {save_path}")
    
    plt.clf()


# 학습 진행 그래프
def plot_training_progress(train_losses, val_precisions, val_recalls, val_f1s, epoch, save_dir):
    """학습 진행 상황 그래프"""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    epochs_range = range(1, len(train_losses) + 1)
    
    # Loss
    axes[0, 0].plot(epochs_range, train_losses, 'b-', linewidth=2, label='Train Loss')
    axes[0, 0].set_xlabel('Epoch', fontsize=12)
    axes[0, 0].set_ylabel('Loss', fontsize=12)
    axes[0, 0].set_title('Training Loss', fontsize=14, fontweight='bold')
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].legend()
    
    # Precision
    axes[0, 1].plot(epochs_range, val_precisions, 'g-', linewidth=2, label='Precision')
    axes[0, 1].set_xlabel('Epoch', fontsize=12)
    axes[0, 1].set_ylabel('Precision', fontsize=12)
    axes[0, 1].set_title('Validation Precision', fontsize=14, fontweight='bold')
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].legend()
    
    # Recall
    axes[1, 0].plot(epochs_range, val_recalls, 'r-', linewidth=2, label='Recall')
    axes[1, 0].set_xlabel('Epoch', fontsize=12)
    axes[1, 0].set_ylabel('Recall', fontsize=12)
    axes[1, 0].set_title('Validation Recall', fontsize=14, fontweight='bold')
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].legend()
    
    # F1 Score
    axes[1, 1].plot(epochs_range, val_f1s, 'm-', linewidth=2, label='F1 Score')
    axes[1, 1].set_xlabel('Epoch', fontsize=12)
    axes[1, 1].set_ylabel('F1 Score', fontsize=12)
    axes[1, 1].set_title('Validation F1 Score', fontsize=14, fontweight='bold')
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].legend()
    
    plt.suptitle(f'Training Progress - Epoch {epoch}', fontsize=16, fontweight='bold')
    plt.tight_layout()
    
    save_path = os.path.join(save_dir, f'training_progress_epoch_{epoch}.png')
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f"✅ 학습 진행 그래프 저장: {save_path}")
    plt.clf()

print("시각화 함수 준비 완료")

In [ ]:
# 메인 학습 루프
train_losses = []
val_precisions = []
val_recalls = []
val_f1s = []

epochs = 10000
best_f1 = 0
save_dir = cfg.work_dir

# 체크포인트 로드 (있는 경우)
checkpoint_path = os.path.join(save_dir, 'last_model.pt')
start_epoch = 0
if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    print(f"체크포인트 로드: {checkpoint_path} (Epoch {start_epoch}부터 시작)")

# Mixed precision scaler
scaler = torch.cuda.amp.GradScaler()

print("=" * 80)
print(f"Point Detection 학습 시작!")
print(f"총 에폭: {epochs}")
print(f"배치 크기: {train_loader.batch_size}")
print(f"훈련 샘플: {len(train_dataset)}")
print(f"검증 샘플: {len(val_dataset)}")
print("=" * 80)

for epoch in range(start_epoch, epochs):
    # 훈련 모드
    model.train()
    
    avg_total_loss = AverageMeter()
    avg_point_loss = AverageMeter()
    avg_class_loss = AverageMeter()
    avg_conf_loss = AverageMeter()
    
    train_pbar = tqdm.tqdm(enumerate(train_loader), total=len(train_loader),
                           desc=f'Epoch {epoch+1}/{epochs} Training')
    
    for i, batch in train_pbar:
        images = batch['images'].to(device).float() / 255.0
        gt_points = batch['points'].to(device).float()
        gt_classes = batch['classes'].to(device).long()
        
        optimizer.zero_grad()
        
        # Forward pass with mixed precision
        with torch.cuda.amp.autocast():
            pred_points, pred_classes, pred_conf = model(images)
            
            # Loss 계산 (Hungarian matching 사용)
            total_loss, point_loss, class_loss, conf_loss = criterion(
                pred_points, pred_classes, pred_conf, gt_points, gt_classes
            )
        
        # Backward pass
        scaler.scale(total_loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        # 손실 업데이트
        avg_total_loss.update(total_loss.item(), images.size(0))
        avg_point_loss.update(point_loss.item(), images.size(0))
        avg_class_loss.update(class_loss.item(), images.size(0))
        avg_conf_loss.update(conf_loss.item(), images.size(0))
        
        # GPU 메모리 동기화
        torch.cuda.synchronize()
        
        # 진행률 표시
        memory = f'{torch.cuda.memory_reserved() / 1E9:.4g}G'
        s = f'Memory: {memory} | Total: {avg_total_loss.avg:.4f} | Point: {avg_point_loss.avg:.4f} | Class: {avg_class_loss.avg:.4f} | Conf: {avg_conf_loss.avg:.4f}'
        train_pbar.set_description(f'Epoch {epoch+1}/{epochs} | {s}')
    
    # 에폭 평균 손실
    train_losses.append(avg_total_loss.avg)
    
    # 검증
    print(f"\n검증 중...")
    precision, recall, f1_score = compute_point_detection_metrics(model, val_loader, device)
    
    val_precisions.append(precision)
    val_recalls.append(recall)
    val_f1s.append(f1_score)
    
    # 결과 출력
    print(f"\nEpoch {epoch+1}/{epochs} Results:")
    print(f"  Train Loss - Total: {avg_total_loss.avg:.4f}, Point: {avg_point_loss.avg:.4f}, Class: {avg_class_loss.avg:.4f}, Conf: {avg_conf_loss.avg:.4f}")
    print(f"  Validation - Precision: {precision:.4f}, Recall: {recall:.4f}, F1-score: {f1_score:.4f}")
    print("-" * 80)
    
    # 베스트 모델 저장 (F1 기준)
    if f1_score > best_f1:
        best_f1 = f1_score
        save_checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': avg_total_loss.avg,
            'point_loss': avg_point_loss.avg,
            'class_loss': avg_class_loss.avg,
            'conf_loss': avg_conf_loss.avg,
            'precision': precision,
            'recall': recall,
            'f1_score': f1_score,
        }
        torch.save(save_checkpoint, os.path.join(save_dir, 'best_model.pt'))
        print(f"🎉 새로운 베스트 모델 저장! F1: {f1_score:.4f}")
    
    # 최신 모델 저장
    last_checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_loss': avg_total_loss.avg,
        'point_loss': avg_point_loss.avg,
        'class_loss': avg_class_loss.avg,
        'conf_loss': avg_conf_loss.avg,
        'precision': precision,
        'recall': recall,
        'f1_score': f1_score,
    }
    torch.save(last_checkpoint, os.path.join(save_dir, 'last_model.pt'))
    
    # 학습 진행 그래프 (10 에폭마다)
    if (epoch + 1) % 10 == 0:
        try:
            print(f"\n📊 Epoch {epoch+1} - 학습 진행 그래프 생성 중...")
            plot_training_progress(train_losses, val_precisions, val_recalls, val_f1s, epoch+1, save_dir)
        except Exception as e:
            print(f"그래프 생성 중 오류: {e}")
    
    # 시각화 (10 에폭마다)
    if (epoch + 1) % 10 == 0:
        try:
            print(f"\n📊 Epoch {epoch+1} - 검증 샘플 시각화:")
            sample_idx = random.randint(0, len(val_dataset) - 1)
            visualize_point_detection(model, val_dataset, idx=sample_idx, epoch=epoch+1, save_dir=save_dir)
        except Exception as e:
            print(f"시각화 중 오류: {e}")

print("\n🎯 학습 완료!")
print(f"최종 베스트 F1 Score: {best_f1:.4f}")
print(f"모델 저장 위치: {save_dir}")
print(f"베스트 모델: {os.path.join(save_dir, 'best_model.pt')}")
print(f"최신 모델: {os.path.join(save_dir, 'last_model.pt')}")

# 최종 성능 요약
if val_f1s:
    final_f1 = val_f1s[-1]
    final_precision = val_precisions[-1]
    final_recall = val_recalls[-1]
    
    print(f"\n📊 최종 성능 요약:")
    print(f"  F1-score: {final_f1:.4f}")
    print(f"  Precision: {final_precision:.4f}")
    print(f"  Recall: {final_recall:.4f}")

In [ ]:
# 데이터 샘플 확인
sample_idx = 0
sample = train_dataset[sample_idx]

print("샘플 데이터 확인:")
print(f"  이미지 shape: {sample['image'].shape}")
print(f"  포인트 개수: {len(sample['points'])}")
print(f"  클래스 개수: {len(sample['classes'])}")
print(f"  포인트 예시 (처음 5개):")
for i in range(min(5, len(sample['points']))):
    x, y = sample['points'][i].tolist()
    cls = sample['classes'][i].item()
    print(f"    Point {i+1}: ({x:.3f}, {y:.3f}), Class: {cls}")

# 시각화
fig, ax = plt.subplots(figsize=(10, 10))
img_display = sample['image'].permute(1, 2, 0).numpy() / 255.0
ax.imshow(img_display)

colors = ['orange', 'green', 'red', 'skyblue', 'blue', 'yellow']
class_names = {
    0: "Neutrophil",
    1: "Epithelial",
    2: "Lymphocyte",
    3: "Plasma",
    4: "Eosinophil",
    5: "Connective tissue"
}

for i in range(len(sample['points'])):
    x_pixel, y_pixel = sample['points'][i].tolist()  # 이미 절대 좌표
    class_id = sample['classes'][i].item()
    ax.scatter(x_pixel, y_pixel, s=100, marker='o',
              facecolors='none', edgecolors=colors[class_id], linewidths=2)

# 범례
legend_elements = [patches.Patch(color=colors[i], label=class_names[i]) 
                  for i in range(len(class_names))]
ax.legend(handles=legend_elements, loc='upper right', fontsize=10)
ax.set_title(f'Training Sample - {len(sample["points"])} points', fontsize=14, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# 📊 데이터 검증: 좌표가 제대로 들어갔는지 확인
print("=" * 80)
print("🔍 데이터 검증 시작")
print("=" * 80)

# 1. 원본 라벨 형식 확인
print("\n1️⃣ 원본 라벨 형식 확인:")
if labels:
    sample_label = labels[0]
    if sample_label:
        print(f"  첫 번째 이미지의 첫 번째 라벨: {sample_label[0]}")
        print(f"  라벨 형식: [class, y, x, h, w]")
        print(f"  - 클래스 (index 0): {sample_label[0][0]}")
        print(f"  - Y 좌표 (index 1): {sample_label[0][1]}")
        print(f"  - X 좌표 (index 2): {sample_label[0][2]}")
        print(f"  - 높이 (index 3): {sample_label[0][3]}")
        print(f"  - 너비 (index 4): {sample_label[0][4]}")

# 2. 데이터셋 샘플 확인
print("\n2️⃣ 데이터셋 변환 후 확인:")
sample = train_dataset[0]
print(f"  이미지 shape: {sample['image'].shape}")
print(f"  포인트 개수: {len(sample['points'])}")
print(f"  클래스 개수: {len(sample['classes'])}")

if len(sample['points']) > 0:
    print(f"\n  처음 3개 포인트 (절대 픽셀 좌표):")
    for i in range(min(3, len(sample['points']))):
        x, y = sample['points'][i].tolist()
        cls = sample['classes'][i].item()
        print(f"    Point {i+1}: x={x:.1f}, y={y:.1f}, class={cls}")

# 3. Collate 함수 테스트
print("\n3️⃣ Collate 함수 테스트 (배치 생성):")
test_batch = next(iter(train_loader))
print(f"  배치 이미지 shape: {test_batch['images'].shape}")
print(f"  배치 포인트 shape: {test_batch['points'].shape}")
print(f"  배치 클래스 shape: {test_batch['classes'].shape}")

# 패딩 확인
for b in range(min(2, test_batch['classes'].shape[0])):
    valid_mask = test_batch['classes'][b] >= 0
    num_valid = valid_mask.sum().item()
    num_padding = (~valid_mask).sum().item()
    print(f"\n  배치 {b+1}:")
    print(f"    - 유효한 포인트: {num_valid}개")
    print(f"    - 패딩: {num_padding}개 (classes=-1)")
    
    if num_valid > 0:
        valid_points = test_batch['points'][b][valid_mask][:3]
        valid_classes = test_batch['classes'][b][valid_mask][:3]
        print(f"    - 처음 3개 유효 포인트:")
        for i in range(len(valid_points)):
            x, y = valid_points[i].tolist()
            cls = valid_classes[i].item()
            print(f"      Point {i+1}: x={x:.4f}, y={y:.4f}, class={cls}")

# 4. 좌표 범위 확인
print("\n4️⃣ 좌표 범위 확인 (0~512 사이여야 함):")
all_points = []
all_classes = []
for i in range(min(10, len(train_dataset))):
    sample = train_dataset[i]
    if len(sample['points']) > 0:
        all_points.extend(sample['points'].tolist())
        all_classes.extend(sample['classes'].tolist())

if all_points:
    all_points = np.array(all_points)
    all_classes = np.array(all_classes)
    print(f"  X 좌표 범위: {all_points[:, 0].min():.4f} ~ {all_points[:, 0].max():.4f}")
    print(f"  Y 좌표 범위: {all_points[:, 1].min():.4f} ~ {all_points[:, 1].max():.4f}")
    print(f"  클래스 분포: {np.bincount(all_classes.astype(int))}")
    
    # 범위 체크
    if all_points[:, 0].min() < 0 or all_points[:, 0].max() > 512:
        print("  ⚠️ 경고: X 좌표가 0~512 범위를 벗어났습니다!")
    if all_points[:, 1].min() < 0 or all_points[:, 1].max() > 512:
        print("  ⚠️ 경고: Y 좌표가 0~512 범위를 벗어났습니다!")
    if 0 <= all_points.min() and all_points.max() <= 512:
        print("  ✅ 모든 좌표가 0~512 범위 내에 있습니다!")

print("\n" + "=" * 80)
print("✅ 데이터 검증 완료!")
print("=" * 80)